# 📑 Legal Document Clause Classifier - Training Pipeline

This notebook provides a complete production-grade pipeline for fine-tuning **Legal-BERT** on the CUAD dataset. It includes EDA, base training, hyperparameter tuning, and detailed evaluation.

### **Workflow:**
1. **Setup**: Libraries & Environment
2. **Data**: Loading & Exploratory Data Analysis (EDA)
3. **Preprocessing**: Tokenization & Label Encoding
4. **Base Training**: Initial Legal-BERT model
5. **Tuning**: Hyperparameter optimization using Optuna
6. **Evaluation**: Metrics, Classification Report & Confusion Matrix
7. **Export**: Saving for Production

## 1. 🛠️ Setup & Libraries

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_recall_curve, average_precision_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments, 
    EarlyStoppingCallback
)
import optuna

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. 📊 Data Loading & EDA

In [ ]:
DATA_PATH = "../../dataset/all_reshaped_clauses.csv"
df = pd.read_csv(DATA_PATH)

print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
# EDA: Distribution of Clause Types
plt.figure(figsize=(12, 10))
sns.countplot(y='clause_type', data=df, order=df['clause_type'].value_counts().index, palette='viridis')
plt.title("Distribution of Legal Clause Types in Dataset")
plt.xlabel("Number of Samples")
plt.ylabel("Clause Type")
plt.show()

print("Top 10 Clause Types:")
print(df['clause_type'].value_counts().head(10))

In [ ]:
# EDA: Sequence Length Analysis
df['text_len'] = df['clause_text'].apply(lambda x: len(str(x).split()))
plt.figure(figsize=(10, 6))
sns.histplot(df['text_len'], bins=50, kde=True, color='blue')
plt.title("Distribution of Clause Word Counts")
plt.xlabel("Word Count")
plt.ylabel("Frequency")
plt.axvline(x=512, color='red', linestyle='--', label='BERT Limit (Approx)')
plt.legend()
plt.show()

In [ ]:
df.columns

## 2.5 🤖 Baseline Algorithm Comparison
Before diving into Legal-BERT, let's see how traditional machine learning algorithms perform on this dataset.

In [ ]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

# Features
X = tfidf.fit_transform(df['clause_text'])

# Labels
y = df['clause_type']

# Train Test Split
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# 1. Logistic Regression
lr = LogisticRegression(max_iter=1000)

lr.fit(X_train_base, y_train_base)

lr_preds = lr.predict(X_test_base)

lr_acc = accuracy_score(y_test_base, lr_preds)

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=100)

rf.fit(X_train_base, y_train_base)

rf_preds = rf.predict(X_test_base)

rf_acc = accuracy_score(y_test_base, rf_preds)

print(f"Logistic Regression Accuracy: {lr_acc:.4f}")
print(f"Random Forest Accuracy: {rf_acc:.4f}")

## 3. ⚙️ Preprocessing

In [ ]:
# Encode Labels
labels = sorted(df['clause_type'].unique())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

df['label'] = df['clause_type'].map(label2id)

# Split Data
train_df, test_df = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.1, stratify=train_df['label'], random_state=42)

print(f"Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}")

In [ ]:
# Tokenization
MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class LegalDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding=True, max_length=512)
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = LegalDataset(train_df['clause_text'], train_df['label'])
val_dataset = LegalDataset(val_df['clause_text'], val_df['label'])
test_dataset = LegalDataset(test_df['clause_text'], test_df['label'])

## 4. 🚀 Base Training

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1_weighted': f1_score(labels, predictions, average='weighted')
    }

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(labels))

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Training Base Model...")
trainer.train()

: 

: 

## 5. 🔍 Evaluation (Before Tuning)

In [ ]:
preds = trainer.predict(test_dataset)
pred_labels = np.argmax(preds.predictions, axis=-1)

print("--- Base Model Performance ---")
print(classification_report(test_df['label'], pred_labels, target_names=labels))

# Confusion Matrix Visualization
cm = confusion_matrix(test_df['label'], pred_labels)
plt.figure(figsize=(15, 12))
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix - Base Model")
plt.show()

In [ ]:
# Algorithm Comparison Graph
bert_acc = trainer.evaluate(test_dataset)['eval_accuracy']
models = ['Logistic Regression', 'Random Forest', 'Legal-BERT']
accuracies = [lr_acc, rf_acc, bert_acc]

plt.figure(figsize=(10, 6))
sns.barplot(x=models, y=accuracies, palette='magma')
plt.ylim(0, 1.0)
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
for i, acc in enumerate(accuracies):
    plt.text(i, acc + 0.02, f"{acc:.2%}", ha='center', fontweight='bold')
plt.show()

## 6. 🧪 Hyperparameter Tuning (Optuna)

In [ ]:
def objective(trial):
    # Parameters to tune
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True)
    batch_size = trial.suggest_categorical("batch_size", [8, 16])
    
    tune_args = TrainingArguments(
        output_dir='./tuning',
        num_train_epochs=5,
        per_device_train_batch_size=batch_size,
        learning_rate=learning_rate,
        eval_strategy="no",
        report_to="none"
    )
    
    tune_trainer = Trainer(
        model=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(labels)),
        args=tune_args,
        train_dataset=train_dataset
    )
    
    tune_trainer.train()
    eval_res = tune_trainer.evaluate(val_dataset)
    return eval_res["eval_loss"]

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=5)

print("Best Hyperparameters:", study.best_params)

## 7. 🏆 Final Training & Export

In [ ]:
# Train with best params
final_args = TrainingArguments(
    output_dir='./final_model',
    num_train_epochs=5,
    per_device_train_batch_size=study.best_params['batch_size'],
    learning_rate=study.best_params['learning_rate'],
    load_best_model_at_end=True,
    eval_strategy="epoch",
    save_strategy="epoch"
)

final_trainer = Trainer(
    model=model,
    args=final_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

final_trainer.train()

# Final Evaluation Graph
history = final_trainer.state.log_history
train_loss = [h['loss'] for h in history if 'loss' in h]
eval_loss = [h['eval_loss'] for h in history if 'eval_loss' in h]

plt.plot(train_loss, label='Train Loss')
plt.plot(eval_loss, label='Val Loss')
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

In [ ]:
# Save as Production Package
OUTPUT_PT = "../ml_models/cuad_classifier.pt"
save_data = {
    "state_dict": model.state_dict(),
    "label2id": label2id,
    "id2label": id2label,
    "model_config": model.config
}
torch.save(save_data, OUTPUT_PT)
print(f"✅ Final Production Model saved to {OUTPUT_PT}")